# Merge layoff pool with confirmed no-layoff pool

Combines `labeled_merged_data_cleaned.csv` (financial statements for tracked companies, labeled by `fix_ticker_conflicts_and_relabel.ipynb`) with `merged_bs_cs_fd_no_layoff.csv` (financial statements for companies confirmed to have no layoff history).

Steps: clean both pools, verify the feature columns match, resolve any column/ticker conflicts, then write a single labeled dataset to `MERGED_DATA["FINAL_MERGED_LABELED_CSV_PATH"]`.

In [10]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import MERGED_DATA

LABELED_LAYOFF_CSV = MERGED_DATA["LABELED_LAYOFF_CSV_PATH"]
NO_LAYOFF_CSV = MERGED_DATA["MERGED_OUTPUT_NO_LAYOFF_CSV_PATH"]
FINAL_OUTPUT_CSV = MERGED_DATA["FINAL_MERGED_LABELED_CSV_PATH"]

ID_COLS = ["company", "date", "quarter"]
LABEL_HELPER_COLS = [
    "same_quarter",
    "next_quarter",
    "layoff_same_quarter",
    "layoff_next_quarter",
    "layoff_same_or_next_quarter",
]

## 1. Load both pools

In [11]:
layoff_df = pd.read_csv(LABELED_LAYOFF_CSV)
no_layoff_df = pd.read_csv(NO_LAYOFF_CSV)

layoff_df["date"] = pd.to_datetime(layoff_df["date"])
no_layoff_df["date"] = pd.to_datetime(no_layoff_df["date"])

print(f"Layoff pool:    {layoff_df.shape}, companies={layoff_df['company'].nunique()}")
print(f"No-layoff pool: {no_layoff_df.shape}, companies={no_layoff_df['company'].nunique()}")

Layoff pool:    (12047, 355), companies=1941
No-layoff pool: (17983, 349), companies=3442


## 2. Clean

Drop exact duplicate rows and check that `(company, date)` is a unique key within each pool.

In [12]:
for name, df in [("layoff", layoff_df), ("no_layoff", no_layoff_df)]:
    before = len(df)
    df.drop_duplicates(inplace=True)
    dupe_keys = df.duplicated(subset=["company", "date"]).sum()
    print(
        f"{name}: dropped {before - len(df)} exact duplicate rows, "
        f"{dupe_keys} remaining duplicate (company, date) keys"
    )
    assert dupe_keys == 0, f"{name} pool has duplicate (company, date) keys"

layoff: dropped 0 exact duplicate rows, 0 remaining duplicate (company, date) keys
no_layoff: dropped 0 exact duplicate rows, 0 remaining duplicate (company, date) keys


## 3. Test: do the feature columns match?

`layoff_df` carries extra label bookkeeping columns from the relabeling notebook, so those are excluded before comparing. Everything else should line up between the two pools.

In [13]:
layoff_feature_cols = set(layoff_df.columns) - set(LABEL_HELPER_COLS) - {"layoff"} - set(ID_COLS)
no_layoff_feature_cols = set(no_layoff_df.columns) - set(ID_COLS)

only_in_layoff = sorted(layoff_feature_cols - no_layoff_feature_cols)
only_in_no_layoff = sorted(no_layoff_feature_cols - layoff_feature_cols)
shared_cols = layoff_feature_cols & no_layoff_feature_cols

print(f"Shared feature columns: {len(shared_cols)}")
print(f"Only in layoff pool ({len(only_in_layoff)}): {only_in_layoff}")
print(f"Only in no-layoff pool ({len(only_in_no_layoff)}): {only_in_no_layoff}")

dtype_conflicts = [
    c for c in shared_cols if layoff_df[c].dtype != no_layoff_df[c].dtype
]
print(f"Dtype conflicts on shared columns: {dtype_conflicts}")

Shared feature columns: 345
Only in layoff pool (1): ['bs_Restricted Common Stock']
Only in no-layoff pool (1): ['cf_Change In Dividend Payable']
Dtype conflicts on shared columns: []


## 4. Resolve column conflicts

Align both pools to the union of feature columns so a plain concat does not need to guess at missing fields — any column absent from one pool is added back as `NaN` there.

In [14]:
union_feature_cols = sorted(layoff_feature_cols | no_layoff_feature_cols)

missing_in_layoff = [c for c in union_feature_cols if c not in layoff_df.columns]
missing_in_no_layoff = [c for c in union_feature_cols if c not in no_layoff_df.columns]

layoff_df = pd.concat(
    [layoff_df, pd.DataFrame(pd.NA, index=layoff_df.index, columns=missing_in_layoff)],
    axis=1,
)
no_layoff_df = pd.concat(
    [no_layoff_df, pd.DataFrame(pd.NA, index=no_layoff_df.index, columns=missing_in_no_layoff)],
    axis=1,
)

assert set(union_feature_cols) <= set(layoff_df.columns)
assert set(union_feature_cols) <= set(no_layoff_df.columns)
print(f"Aligned both pools to {len(union_feature_cols)} feature columns")

Aligned both pools to 347 feature columns


## 5. Resolve ticker conflicts

A ticker should not live in both the layoff-tracked pool and the "confirmed no layoff" pool — the latter is supposed to be a clean negative set. Any overlap is dropped from the no-layoff pool since that company's financials (and label) already exist in the layoff pool.

In [15]:
overlap_tickers = sorted(set(layoff_df["company"]) & set(no_layoff_df["company"]))
print(f"Tickers present in both pools ({len(overlap_tickers)}): {overlap_tickers}")

if overlap_tickers:
    flagged_as_layoff = (
        layoff_df.loc[layoff_df["company"].isin(overlap_tickers), "layoff"].eq(1).any()
    )
    print(f"Any overlapping ticker labeled as an actual layoff quarter? {flagged_as_layoff}")

    no_layoff_df = no_layoff_df.loc[~no_layoff_df["company"].isin(overlap_tickers)].copy()
    print(f"Dropped {len(overlap_tickers)} overlapping tickers from the no-layoff pool")

assert not (set(layoff_df["company"]) & set(no_layoff_df["company"])), "ticker overlap remains"

Tickers present in both pools (2): ['STNE', 'TEAD']
Any overlapping ticker labeled as an actual layoff quarter? False
Dropped 2 overlapping tickers from the no-layoff pool


## 6. Build the combined labeled dataset

Every row from the no-layoff pool is label `0` by construction. `layoff_this_quarter` / `layoff_next_quarter` break the `layoff` label down by timing (renamed from the source file's `layoff_same_quarter`), and `source_pool` is kept for traceability so it is easy to trace a row back to where its financials came from.

In [16]:
layoff_df["layoff_this_quarter"] = layoff_df["layoff_same_quarter"]
no_layoff_df["layoff_this_quarter"] = 0
no_layoff_df["layoff_next_quarter"] = 0

layoff_df["source_pool"] = "layoff_tracked"
no_layoff_df["source_pool"] = "confirmed_no_layoff"
no_layoff_df["layoff"] = 0

final_cols = ID_COLS + union_feature_cols + [
    "layoff_this_quarter",
    "layoff_next_quarter",
    "layoff",
    "source_pool",
]

combined = pd.concat(
    [layoff_df[final_cols], no_layoff_df[final_cols]],
    ignore_index=True,
)

print(combined.shape)

(30018, 354)


C:\Users\phuon\AppData\Local\Temp\ipykernel_4688\605496242.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  layoff_df["layoff_this_quarter"] = layoff_df["layoff_same_quarter"]
C:\Users\phuon\AppData\Local\Temp\ipykernel_4688\605496242.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  layoff_df["source_pool"] = "layoff_tracked"


## 7. Validate the merge

In [17]:
assert combined.duplicated(subset=["company", "date"]).sum() == 0, "duplicate (company, date) rows"
assert set(combined.columns) == set(final_cols)
assert combined["layoff"].isin([0, 1]).all()
assert combined["layoff_this_quarter"].isin([0, 1]).all()
assert combined["layoff_next_quarter"].isin([0, 1]).all()

print("Companies:", combined["company"].nunique())
print(combined[["layoff_this_quarter", "layoff_next_quarter", "layoff"]].sum())
print(combined["layoff"].value_counts())
print(combined["source_pool"].value_counts())

Companies: 5381
layoff_this_quarter     712
layoff_next_quarter     656
layoff                 1205
dtype: int64
layoff
0    28813
1     1205
Name: count, dtype: int64
source_pool
confirmed_no_layoff    17971
layoff_tracked         12047
Name: count, dtype: int64


## 8. Save

In [18]:
combined.to_csv(FINAL_OUTPUT_CSV, index=False)
print(f"Wrote {len(combined)} rows to {FINAL_OUTPUT_CSV}")

Wrote 30018 rows to c:\Users\phuon\Desktop\osint-proj\layoff-detector\data\processed\final_merged_labeled_dataset.csv
